# Stage1 XSD−SOXX 行业广度 H5 单坐标信息检验

原 Stage1 39 项冻结不变；候选只加入截至信号日的 XSD−SOXX 20 日总收益差，预期系数为正。历史分红/拆分版本未获首次公开时钟，所有结果仅为条件性开发反馈。

In [1]:
import os,json,hashlib,sys,sklearn
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd();card=root/'research/stage1_xsd_breadth_h5_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and sys.version_info[:2]==(3,12) and sklearn.__version__=='1.9.1'
parent=plan['parent'];parent_card=root/parent['source_card']
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:assert sha(parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/name)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
spec=plan['source'];assert sha(root/spec['summary'])==spec['summary_sha256'];source_summary=json.loads((root/spec['summary']).read_text())
assert source_summary['recorder_id']==spec['recorder_id'] and source_summary['nas_readback']['verified']
source_artifact=Path(source_summary['recorder_uri'])/source_summary['experiment_id']/source_summary['recorder_id']/'artifacts'/spec['feature_artifact']
assert sha(source_artifact)==spec['mapped_artifact_sha256'], 'Qlib mapped source object SHA mismatch'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(source_summary['recorder_uri']):mapped=R.get_recorder(recorder_id=source_summary['recorder_id'],experiment_id=source_summary['experiment_id']).load_object(spec['feature_artifact'])
assert len(mapped)==3964 and mapped.index.is_unique and mapped.source_known.all()
feature=mapped['XSD_minus_SOXX_log_total_return_20'].astype(float).reindex(frame.index).rename(plan['single_mechanism']['feature'])
forward=frame.soxx_forward_open_return.astype(float);parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert train_dates.sum()==plan['cohorts']['expected_mature_train_rows'] and held_dates.sum()==plan['cohorts']['expected_mature_held_rows']
assert feature.loc[train_dates|held_dates].notna().all() and np.isfinite(feature.loc[train_dates|held_dates]).all()
augmented=base[base_names].copy();augmented[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
train_prepared=dataset.prepare('train',col_set=['feature','label','meta']);held_prepared=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'Stage1_XSD_breadth_DatasetH_label_maturity_preflight_passed_conditional_source_no_fit','plan_sha256':sha(card/'plan.json'),'parent_recorder_id':parent['recorder_id'],'source_recorder_id':source_summary['recorder_id'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_unknown_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(feature.loc[held_dates].isna().sum()),'training_feature_min':float(feature.loc[train_dates].min()),'training_feature_max':float(feature.loc[train_dates].max()),'source_Recorder_object_loads':5,'DatasetH_builds':1,'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False,'historical_corporate_action_vintage_qualified':False}
receipt=card/'preflight_result.json'
if receipt.exists():assert json.loads(receipt.read_text())==report
else:receipt.write_text(json.dumps(report,indent=2)+'\n')
print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_unknown_rows','candidate_held_unknown_rows','training_feature_min','training_feature_max')})


2026/10/23 04:11:01 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[78152:MainThread](2026-10-23 04:11:01,563) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[78152:MainThread](2026-10-23 04:11:01,565) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[78152:MainThread](2026-10-23 04:11:01,565) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[78152:MainThread](2026-10-23 04:11:02,953) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[78152:MainThread](2026-10-23 04:11:02,954) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[78152:MainThread](2026-10-23 04:11:02,954) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'status': 'Stage1_XSD_breadth_DatasetH_label_maturity_preflight_passed_conditional_source_no_fit', 'training_mature_rows': 3014, 'held_mature_rows': 940, 'candidate_train_unknown_rows': 0, 'candidate_held_unknown_rows': 0, 'training_feature_min': -0.07606804500765295, 'training_feature_max': 0.0624089601077847}


## 独立设计审阅门

无拟合预检通过后，独立审阅准确的计划和两个代码单元，再执行固定两次 Ridge 和一次区块检验。

In [2]:
import shlex,tarfile,tempfile,subprocess
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
review_path=card/'design_review_final.json';assert review_path.exists()
review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());preflight_source=''.join(notebook['cells'][1]['source']);preflight_hash=hashlib.sha256(preflight_source.encode()).hexdigest();fit_source=''.join(notebook['cells'][3]['source']);fit_hash=hashlib.sha256(fit_source.encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['preflight_sha256']==sha(card/'preflight_result.json') and review['preflight_cell_source_sha256']==preflight_hash and review['fit_cell_source_sha256']==fit_hash
assert report==json.loads((card/'preflight_result.json').read_text()) and not report['historical_corporate_action_vintage_qualified']
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists()
state={'plan_sha256':sha(card/'plan.json'),'fit_cell_source_sha256':fit_hash,'fit_attempts':[],'fits_done':[],'prediction_attempts':[],'predictions_done':[],'bootstrap_attempted':False,'new_native_accounts':0}
state_path.write_text(json.dumps(state,indent=2)+'\n')
tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)];he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows']==3014 and len(he)==report['held_mature_rows']==940
yname='soxx_forward_five_open_log_return';feature_name=plan['single_mechanism']['feature'];ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
inputs={'baseline':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),'candidate':(tr['feature'][base_names+[feature_name]].to_numpy(float),he['feature'][base_names+[feature_name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).all() and np.isfinite(inputs['candidate'][1][:,-1]).all()
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));state_path.write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True)
 recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,derived_coordinate=feature,source_hashes={'xsd_breadth':spec['summary_sha256'],'original39':parent['feature_artifact_sha256']})
 assert len(recorder.load_object('inputs/dataset').prepare('train',col_set=['feature','label','meta']))==report['train_segment_rows']
 for family in ('baseline','candidate'):
  assert family not in state['fit_attempts'] and len(state['fit_attempts'])<plan['budget']['model_fit_attempts_max']
  state['fit_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
  recorder.save_objects(artifact_path='models',**{family:model});models[family]=model;state['fits_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  state['prediction_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  prediction=model.predict(inputs[family][1]);assert np.isfinite(prediction).all()
  recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})});predictions[family]=prediction
  state['predictions_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n');recorder.log_metrics(completed_fits=len(state['fits_done']),completed_predictions=len(state['predictions_done']))
 assert len(state['fits_done'])==len(state['predictions_done'])==2
 dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2;daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2;daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
 assert daily.signal_date.is_monotonic_increasing and len(daily)==940
 halves=[daily.loc[daily.signal_date.lt(pd.Timestamp('2025-01-01'))],daily.loc[daily.signal_date.ge(pd.Timestamp('2025-01-01'))]];assert min(map(len,halves))>300
 bootstrap=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempted']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
 interval,_,replicates=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],alpha=bootstrap['one_sided_alpha'])
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
 metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),'candidate_held_mse':float(daily.candidate_squared_error.mean()),'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],'candidate_standardized_coordinate_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),'bootstrap_lower_positive':metrics['bootstrap_lower']>0};passed=bool(all(gates.values()))
 daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str);quarters=[{'quarter':q,'sessions':len(g),'paired_mean_mse_improvement':float(g.paired_mse_improvement.mean())} for q,g in daily.groupby('quarter',sort=True)]
 recorder.save_objects(artifact_path='analysis/daily',held=daily);recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,bootstrap_replicates=replicates,information_gate_pass=passed)
 recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':'conditional_development_XSD_breadth_information_pass_action_vintage_pending_no_account' if passed else 'not_promoted_fixed_XSD_minus_SOXX_breadth_H5_information_failed','objective_achieved':False,'information_gate_pass':passed,'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'historical_corporate_action_vintage_qualified':False,'reused_market_history':True,'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'parent_feature_sha256':parent['feature_artifact_sha256'],'source_summary_sha256':spec['summary_sha256'],'artifact_policy':plan['artifact_policy'],'limitations':['Current Yahoo XSD/SOXX corporate-action and adjusted-close revisions lack exact historical first-public certification','Information MSE is not fee-net SOXX funded account result','Reused held 2023-2026 is development feedback, not independent forward']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'result_review.json').write_text(json.dumps({'status':summary['decision'],'plan_sha256':summary['plan_sha256'],'recorder_id':state['recorder_id'],'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0},indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='credit-info-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/models/baseline') for name in names) and any(name.endswith('/artifacts/models/candidate') for name in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['completed']=True;state['nas_archive_sha256']=evidence['sha256'];state_path.write_text(json.dumps(state,indent=2)+'\n')
print({k:summary[k] for k in ('decision','information_gate_pass','new_model_fits','new_native_accounts','recorder_id')});print(metrics)


[78152:MainThread](2026-10-23 04:11:02,971) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_xsd_breadth_h5_information_20261023.


[78152:MainThread](2026-10-23 04:11:02,972) INFO - qlib.workflow - [exp.py:258] - Experiment 866598635718787618 starts running ...


[78152:MainThread](2026-10-23 04:11:03,061) INFO - qlib.workflow - [recorder.py:345] - Recorder 4340e11b06fa4c238604982d68fd9b3a starts running under Experiment 866598635718787618 ...


[78152:MainThread](2026-10-23 04:11:05,965) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'decision': 'not_promoted_fixed_XSD_minus_SOXX_breadth_H5_information_failed', 'information_gate_pass': False, 'new_model_fits': 2, 'new_native_accounts': 0, 'recorder_id': '4340e11b06fa4c238604982d68fd9b3a'}
{'baseline_held_mse': 0.0025262669365982205, 'candidate_held_mse': 0.002529822407478728, 'held_paired_mse_improvement': -3.555470880506835e-06, 'held_half_paired_improvements': [-1.8042017216110384e-05, 1.3047831084043351e-05], 'candidate_standardized_coordinate_coefficient': 0.0006450257021746164, 'bootstrap_lower': -2.7555290557740157e-05, 'bootstrap_upper': 2.283176112236068e-05, 'bootstrap': {'paired_mean_loss_improvement': -3.555470880506835e-06, 'lower_bound': -2.7555290557740157e-05, 'upper_bound': 2.283176112236068e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 940, 'scored_rows': 940}}


# Stage1 XSD−SOXX 行业广度 H5 信息检验

结论：`not_promoted`。冻结 Stage1 39 项输入加一个 XSD−SOXX 20 源交易日总收益差，对完整复用历史留出段没有增量信息；固定版本关闭，不改符号、窗口、ETF 或标签补救。

原生 Qlib DatasetH 训练成熟样本 3,014 条，留出成熟样本 940 条；两次固定 Ridge 拟合，一次 10,000 次成对平稳区块抽样（单侧 alpha=0.025/29），新增账户 0。Qlib Recorder `4340e11b06fa4c238604982d68fd9b3a`；NAS 归档回读 SHA-256 `4053a77399a02e71d648d3a993323dd2fbb3b67b298e5a1138fe64bdeb3d6d66`。

留出 MSE：原 Stage1 0.002526267，候选 0.002529822；成对改善 -0.000003555。新增标准化系数 0.000645026 为正，满足方向门；前后两半成对改善分别为 -0.000018042 与 0.000013048，全段与双半段门失败；校正区块下界 -0.000027555，不为正。

区间归因：2024Q1（−0.00007137）与 2024Q3（−0.00003460）明显变差，2026Q2（+0.00007654）局部改善。2024Q3 是原 Stage1 现金保护有效的季度，候选信息却变差；它没有可靠区分应保护与应参与的状态。2026Q4 仅 1 条样本，不作独立判断。这些事后季度仅用于解释失败，不生成切换阈值。

限制：XSD/SOXX 今日回溯的拆分重述价格和除息额缺少当年的首次公开/修订版本；940 条留出历史已被反复用于开发，不是独立验证。信息 MSE 也不是费用后对 SOXX 的账户收益。

下一步：回到冻结 Stage1 完整账户的现金机会损失与保护成功区间，寻找与行业内部广度、信用动量、静态持仓集中度不同的可事前获得机制。先核对来源与已有输入，再冻结单一信息检验；通过后才设计 0bp 原生账户。